# Particle in a box and the colour of dyes

*physchem · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/physchem/particle_box_dyes)

Why do longer conjugated molecules absorb redder light? The $\pi$ electrons of a cyanine dye move freely along the chain, like particles in a one-dimensional box of length $L$. Filling the levels $E_n \propto n^2/L^2$ two by two gives a HOMO→LUMO gap $\Delta E = h^2(N+1)/8m_eL^2$ and a predicted $\lambda_{max}$ for a whole homologous series at once; a batched finite-difference Hamiltonian, diagonalised with torch.linalg.eigh, reproduces the analytic levels.

**Vector form**

$$E_n = \frac{h^2 n^2}{8 m_e L^2},\qquad L = (n_{\mathrm{bonds}} + 2)\,l,\qquad \Delta E = E_{N/2+1} - E_{N/2} = \frac{h^2 (N+1)}{8 m_e L^2} = \frac{hc}{\lambda_{max}},\qquad -\frac{\hbar^2}{2m_e}\,\psi' + V\psi = E\,\psi$$

**Index (tensor) form**

$$E_{bn} = \frac{h^2}{8m_e}\,\frac{n_n n_n}{L_b L_b} \qquad \lambda_b = \frac{8 m_e c\,L_b^2}{h\,(N_b+1)} \qquad H_{bij} = \frac{\hbar^2}{2m_e\,\Delta x_b^2}\,(2\delta_{ij} - \delta_{i,j+1} - \delta_{i+1,j}) + V_{bi}\,\delta_{ij} \qquad H_{bij}\,\psi_{bjn} = E_{bn}\,\psi_{bin}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $N_b,\ n^{\mathrm{bonds}}_b$ | `molecules(series, n_max)` | (B,), (B,) | π electrons and conjugated bonds of molecule $b$ in the series |
| $L_b$ | `box_length(n_bonds, bond, end)` | (B,) | box length: all bonds plus one bond length past each end atom |
| $E_{bn}$ | `H2_8M * einsum('n,b->bn', n**2, 1/L**2)` | (B, n) | analytic levels; no index summed, an outer product of $n^2$ and $1/L^2$ |
| $\Delta E_b,\ \lambda_b$ | `homo_lumo_gap(N, L), HC / dE` | (B,) | HOMO $n = N/2$ to LUMO $n = N/2 + 1$, and the absorbed wavelength |
| $x_{bi},\ \Delta x_b$ | `grids(L, n_points)` | (B, M), (B,) | interior grid points of each box; $\psi = 0$ on the walls |
| $H_{bij}$ | `HBAR2_2M * einsum('b,ij->bij', 1/dx**2, D) + diag_embed(V)` | (B, M, M) | one finite-difference Hamiltonian per molecule, stacked |
| $H_{bij}\psi_{bjn} = E_{bn}\psi_{bin}$ | `torch.linalg.eigh(H)` | (B, M), (B, M, M) | repeated $j$ ⇒ summed; eigh diagonalises the whole batch in one call |
| $c_{bk}$ | `einsum('bw,wk->bk', 1 - A, rgb)` | (B, 3) | repeated $w$ ⇒ summed over visible wavelengths: the colour left after absorption |

Kuhn's free-electron model (1948) treats the $\pi$ electrons of a conjugated chain as independent particles in a box. In a cyanine dye $\mathrm{R_2N{-}(CH{=}CH)_k{-}CH{=}N^+R_2}$ the positive charge is shared by the two nitrogens, so all the chain bonds are nearly equal (about $l = 0.139$ nm) and the flat-bottomed box is a good picture. With $2k + 2$ bonds, $N = 2k + 4$ electrons and the box sticking out one bond past each nitrogen, $L = (2k + 4)l$ and $\lambda_{max} = 8m_ec\,L^2/h(N + 1)$ grows almost linearly with $k$: the model predicts about 125 nm per added CH=CH unit, and the red squares (rounded measured values for the simplest cyanines) climb by about 100 nm. This is the classic undergraduate experiment: record the visible spectra of a few cyanine dyes of increasing chain length, then use $\lambda_{max}$ to fit the box length or the end extension. The numerical solver gives the same levels to better than $10^{-3}$ (its error falls as $(n\,\Delta x)^2$), and it can do what the formula cannot: switch on a potential. Polyenes have alternating double and single bonds; in the plain box their predicted $\lambda_{max}$ runs far into the infrared, whereas real polyenes level off (β-carotene, with 11 C=C, absorbs near 450 nm). Choose the polyene series and raise $V_{alt}$ to 1–2 eV: the periodic potential opens a gap that stays finite for long chains, which is the one-dimensional seed of a band gap. The colour swatches show white light minus a Gaussian absorption band at each $\lambda_{max}$; a dye that absorbs yellow looks blue, and one that absorbs only UV looks colourless.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/physchem/particle_box_dyes/main.py
"""Main loop: parameters → molecules and boxes → analytic and numerical levels → λ_max and colours → plot."""

from pathlib import Path

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    bond, end = params["bond"], params["end"]
    names, n_bonds, n_el = tensors.molecules(params["series"], params["n_max"], dev)  # B names, (B,), (B,)
    L = tensors.box_length(n_bonds, bond, end)  # (B,) nm
    n_levels = int(n_el.max()) // 2 + 2  # one above the highest LUMO
    n = tensors.level_numbers(n_levels, dev)  # (n,)

    # analytic free-electron model
    E_exact = equation.energy_levels(n, L)  # (B, n) eV
    gap = equation.homo_lumo_gap(n_el, L)  # (B,) eV
    lam = equation.wavelength(gap)  # (B,) nm

    # numerical box: finite-difference Hamiltonian, diagonalised for all molecules at once
    n_points = 120 if fast_mode() else params["n_points"]
    x, dx = tensors.grids(L, n_points)  # (B, M), (B,)
    V = equation.alternation_potential(x, params["V_alt"], bond, end)  # (B, M)
    E_num, psi = solve.solve(equation.hamiltonian(dx, V), dx, n_levels)  # (B, n), (B, M, n)
    homo = (n_el // 2 - 1)[:, None]  # 0-based column of the HOMO
    gap_num = (E_num.gather(1, homo + 1) - E_num.gather(1, homo))[:, 0]
    lam_num = equation.wavelength(gap_num)

    # colour of each dye: white light minus its absorption band
    lam_vis = tensors.visible_wavelengths(dev)  # (W,)
    rgb = equation.wavelength_rgb(lam_vis)  # (W, 3)
    colour = equation.perceived_colour(equation.absorption_band(lam_vis, lam_num), rgb)  # (B, 3)

    print(
        f"series {params['series']}   ·   bond l = {bond:.3f} nm   ·   end extension {end:g} l   ·   "
        f"V_alt = {params['V_alt']:g} eV"
    )
    print(f"{'molecule':<11} {'bonds':>5} {'N_π':>4} {'L (nm)':>7} {'ΔE (eV)':>8} {'λ_max (nm)':>11} {'numeric':>8}")
    for b, name in enumerate(names):
        print(
            f"{name:<11} {int(n_bonds[b]):>5} {int(n_el[b]):>4} {L[b].item():7.3f} {gap[b].item():8.3f} "
            f"{lam[b].item():11.1f} {lam_num[b].item():8.1f}"
        )
    if params["V_alt"] == 0:
        ratio = E_num[-1, :6] / E_num[-1, 0]
        err = ((E_num - E_exact) / E_exact).abs().max().item()
        print(f"numerical E_n/E_1 (longest box) = {', '.join(f'{r:.3f}' for r in ratio.tolist())}   (exact n²)")
        print(f"largest relative error of the {n_points}-point grid: {err:.1e}")
    shift = (lam[1:] - lam[:-1]).mean().item()
    print(f"each added CH=CH unit shifts λ_max by {shift:.0f} nm on average")

    measured = tensors.measured_cyanine(dev) if params["series"] == "cyanine" else None
    return plot.animate(
        names,
        L,
        n_bonds,
        bond,
        end,
        n_el,
        E_num,
        psi,
        x,
        lam,
        lam_num,
        colour,
        measured,
        out_dir / "particle_box_dyes.gif",
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/physchem/particle_box_dyes/equation.py
r"""Physics → tensor translation.

Free-electron model: the π electrons of a conjugated chain move freely along it, as particles in a
one-dimensional box of length L. For molecule b and level n

    E_{bn} = h² n² / (8 m_e L_b²)

N π electrons fill the lowest N/2 levels two by two (Pauli), so the HOMO is n = N/2 and the LUMO n = N/2 + 1.
The longest-wavelength absorption lifts one electron across that gap:

    ΔE_b = E_{b, N/2+1} − E_{b, N/2} = h² (N_b + 1) / (8 m_e L_b²),      λ_b = h c / ΔE_b

Numerically the same box is a matrix: on M interior grid points

    H_{bij} = (ħ²/2m_e) (2 δ_{ij} − δ_{i,j+1} − δ_{i+1,j}) / dx_b²  +  V_{bi} δ_{ij}
    H_{bij} ψ_{bjn} = E_{bn} ψ_{bin}          (j summed)

 Units: eV and nm.
"""

import math

import torch

# SI constants → eV and nm
H_PLANCK = 6.62607015e-34  # J s
M_E = 9.1093837015e-31  # kg
EV = 1.602176634e-19  # J
H2_8M = H_PLANCK**2 / (8 * M_E) / EV * 1e18  # h²/(8 m_e) = 0.376 eV nm²
HBAR2_2M = H2_8M / math.pi**2  # ħ²/(2 m_e) = h²/(8 π² m_e) = 0.0381 eV nm²
HC = 1239.84198  # h c in eV nm


def energy_levels(n: torch.Tensor, L: torch.Tensor) -> torch.Tensor:
    """E_bn = h² n_n² / (8 m L_b²).   n: (n,), L: (B,) → (B, n) eV.   No index summed: an outer product."""
    return H2_8M * torch.einsum("n,b->bn", n**2, 1.0 / L**2)


def homo_lumo_gap(n_electrons: torch.Tensor, L: torch.Tensor) -> torch.Tensor:
    """ΔE_b = h² (N_b + 1) / (8 m L_b²)   (B,) eV — the difference of (N/2+1)² and (N/2)²."""
    return H2_8M * (n_electrons + 1) / L**2


def wavelength(delta_E: torch.Tensor) -> torch.Tensor:
    """λ = h c / ΔE   (nm)."""
    return HC / delta_E


def alternation_potential(x: torch.Tensor, V0: float, bond: float, end: float) -> torch.Tensor:
    """V_bi = −(V0/2) cos(π (x_bi − x₀)/l): low on double bonds, high on single bonds   (B, M) eV.

    x₀ = (end + ½) l is the middle of the first bond. Its period, 2l, is one double + one single bond.
    V0 = 0 is the plain box; a polyene's bond alternation is better described with V0 ≈ 1–2 eV.
    """
    x0 = (end + 0.5) * bond
    return -0.5 * V0 * torch.cos(math.pi * (x - x0) / bond)


def hamiltonian(dx: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    """H_bij = (ħ²/2m)(2δ_ij − δ_i,j+1 − δ_i+1,j)/dx_b² + V_bi δ_ij.   dx: (B,), V: (B, M) → (B, M, M)."""
    M = V.shape[1]
    eye = torch.eye(M, device=V.device, dtype=V.dtype)
    second_diff = 2 * eye - torch.diag(torch.ones(M - 1, device=V.device, dtype=V.dtype), 1)
    second_diff = second_diff - torch.diag(torch.ones(M - 1, device=V.device, dtype=V.dtype), -1)  # −dx² d²/dx²
    kinetic = HBAR2_2M * torch.einsum("b,ij->bij", 1.0 / dx**2, second_diff)
    return kinetic + torch.diag_embed(V)


def absorption_band(lam: torch.Tensor, lam_max: torch.Tensor, width: float = 40.0) -> torch.Tensor:
    """A_bw = exp(−½ ((λ_w − λmax_b)/σ)²): a Gaussian absorption band of each dye   (B, W)."""
    return torch.exp(-0.5 * ((lam[None, :] - lam_max[:, None]) / width) ** 2)


def wavelength_rgb(lam: torch.Tensor) -> torch.Tensor:
    """An approximate RGB colour (W, 3) for each visible wavelength (piecewise-linear rainbow, dimmed at the ends)."""
    r = torch.where(lam < 440, (440 - lam) / 60, torch.where(lam < 510, 0.0 * lam, (lam - 510) / 70)).clamp(0, 1)
    g = torch.where(lam < 490, (lam - 440) / 50, torch.where(lam < 580, 1.0 + 0.0 * lam, (645 - lam) / 65)).clamp(0, 1)
    b = torch.where(lam < 490, 1.0 + 0.0 * lam, (510 - lam) / 20).clamp(0, 1)
    fade = torch.where(
        lam < 420, 0.3 + 0.7 * (lam - 380) / 40, torch.where(lam > 700, 0.3 + 0.7 * (780 - lam) / 80, 1.0)
    )
    return torch.stack([r, g, b], dim=-1) * fade.clamp(0, 1)[:, None]


def perceived_colour(A: torch.Tensor, rgb: torch.Tensor) -> torch.Tensor:
    """Colour of the light that is NOT absorbed:  c_bk = Σ_w (1 − A_bw) rgb_wk / Σ_w rgb_wk   (B, 3).

    Repeated w ⇒ summed over the visible spectrum; white light minus the absorbed band gives the dye's colour.
    """
    transmitted = torch.einsum("bw,wk->bk", 1.0 - A, rgb)
    return (transmitted / rgb.sum(0)).clamp(0, 1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/physchem/particle_box_dyes/tensors.py
"""Build the concrete tensors: the batch of molecules, their box lengths, the level numbers and the grids.

Units: lengths in nm, energies in eV, wavelengths in nm.
"""

import torch


def molecules(series: str, n_max: int, device: torch.device) -> tuple[list[str], torch.Tensor, torch.Tensor]:
    """Names, number of conjugated bonds (B,) and number of π electrons (B,) for a homologous series.

    cyanine  R₂N–(CH=CH)ₖ–CH=N⁺R₂,  k = 0 … n_max:  2k+3 chain atoms, 2k+2 bonds, 2k+4 π electrons
             (every carbon gives one π electron, the two nitrogens three between them).
    polyene  H–(CH=CH)ₖ–H,          k = 2 … n_max+2: 2k carbons, 2k−1 bonds, 2k π electrons.
    """
    if series == "cyanine":
        k = torch.arange(0, n_max + 1, device=device)
        names = [f"{2 * int(i) + 1}-methine" for i in k]  # number of CH groups between the two N atoms
        return names, 2 * k + 2, 2 * k + 4
    k = torch.arange(2, n_max + 3, device=device)
    names = [f"{int(i)} C=C" for i in k]
    return names, 2 * k - 1, 2 * k


def box_length(n_bonds: torch.Tensor, bond: float, end: float) -> torch.Tensor:
    """L_b = (n_bonds + 2·end) · l   (B,) nm.

    Kuhn's choice: the box runs over every conjugated bond and sticks out by `end` bond lengths
    past each terminal atom, because the π cloud does not stop dead at the last nucleus.
    """
    return (n_bonds.to(torch.get_default_dtype()) + 2.0 * end) * bond


def level_numbers(n_levels: int, device: torch.device) -> torch.Tensor:
    """n = 1, 2, …, n_levels   (n,)."""
    return torch.arange(1, n_levels + 1, device=device, dtype=torch.get_default_dtype())


def grids(L: torch.Tensor, n_points: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Interior grid points x_bi (B, M) of each box [0, L_b] and the spacing dx_b (B,).

    ψ = 0 at the two walls, so only the M interior points are unknowns.
    """
    dx = L / (n_points + 1)  # (B,)
    i = torch.arange(1, n_points + 1, device=L.device, dtype=L.dtype)  # (M,)
    return torch.einsum("b,i->bi", dx, i), dx


def visible_wavelengths(device: torch.device) -> torch.Tensor:
    """λ grid over the visible range, 380–780 nm   (W,)."""
    return torch.linspace(380.0, 780.0, 201, device=device)


def measured_cyanine(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Approximate measured λ_max (nm) of the simple cyanines Me₂N–(CH=CH)ₖ–CH=N⁺Me₂ for k = 1 … 4.

    Rounded literature values (about 100 nm per added CH=CH), shown only as a rough check of the model.
    """
    k = torch.tensor([1, 2, 3, 4], device=device)
    lam = torch.tensor([313.0, 416.0, 519.0, 625.0], device=device)
    return k, lam


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/physchem/particle_box_dyes/solve.py
"""Diagonalise the finite-difference box Hamiltonian of every molecule at once with torch.linalg.eigh."""

import torch


def solve(H: torch.Tensor, dx: torch.Tensor, n_levels: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Lowest levels E (B, n) and normalised wavefunctions ψ (B, M, n) of the batched Hamiltonian H (B, M, M).

    eigh treats the leading dimension as a batch: one call diagonalises every box.
    Column n of ψ_b is state n + 1; it is scaled so that Σ_i ψ_bin² dx_b = 1.
    """
    # float64: the largest grid eigenvalue is ~10⁵ × the lowest, too much spread for float32 round-off
    E, psi = torch.linalg.eigh(H.double())  # ascending energies
    E, psi = E[:, :n_levels].to(H.dtype), psi[:, :, :n_levels].to(H.dtype)
    psi = psi / torch.sqrt(dx)[:, None, None]  # unit vectors → Σ ψ² dx = 1
    sign = torch.sign(psi[:, :1, :])  # fix the arbitrary sign: every state starts upward at the left wall
    return E, psi * sign


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/physchem/particle_box_dyes/plot.py
"""Animate the series: each box with its filled levels and HOMO→LUMO jump, λ_max against chain length, colours."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle

from problems._common.plotting import frame_indices, save_gif

FRAMES_PER_MOLECULE = 10


def _rgb(lam_nm: float) -> tuple[float, float, float]:
    """Display colour for one wavelength (grey outside the visible range)."""
    if not 380 <= lam_nm <= 780:
        return (0.42, 0.45, 0.5)
    from .equation import wavelength_rgb

    r, g, b = wavelength_rgb(torch.tensor([lam_nm])).squeeze(0).tolist()
    return (0.85 * r, 0.85 * g, 0.85 * b)  # a little darker so pale yellows stay visible on white


def animate(names, L, n_bonds, bond, end, n_el, E_num, psi, x, lam, lam_num, colour, measured, path: Path) -> Path:
    L, n_bonds, n_el, E, psi, x = (
        L.cpu().numpy(),
        n_bonds.cpu().numpy(),
        n_el.cpu().numpy(),
        E_num.cpu().numpy(),
        psi.cpu().numpy(),
        x.cpu().numpy(),
    )
    lam, lam_num, colour = lam.cpu().numpy(), lam_num.cpu().numpy(), colour.cpu().numpy()
    B = len(names)
    homo = n_el // 2 - 1  # 0-based index of the HOMO
    e_top = max(E[b, homo[b] + 1] for b in range(B)) * 1.12

    fig = plt.figure(figsize=(8.4, 4.4))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.15, 1], height_ratios=[1.6, 1])
    ax_box = fig.add_subplot(grid[:, 0])
    ax_lam = fig.add_subplot(grid[0, 1])
    ax_sw = fig.add_subplot(grid[1, 1])

    # --- λ_max against chain length ---
    k = np.arange(B)
    ax_lam.imshow(
        _rainbow(), extent=(-0.6, B - 0.4, 380, 780), aspect="auto", origin="lower", alpha=0.22, zorder=0
    )  # the visible band
    ax_lam.plot(k, lam, "-", color="#374151", lw=1.4, label="analytic  $h^2(N+1)/8mL^2$")
    ax_lam.plot(k, lam_num, "o", ms=4.5, mfc="none", color="#2563eb", label="numerical eigh")
    if measured is not None:
        km, lm = (t.cpu().numpy() for t in measured)
        ax_lam.plot(km, lm, "s", ms=4, color="#dc2626", label="measured (approx.)")
    (mark,) = ax_lam.plot([], [], "o", ms=9, color="#f59e0b", zorder=5)
    ax_lam.set_xlim(-0.6, B - 0.4)
    ax_lam.set_ylim(min(150, lam.min() * 0.9), max(820, lam.max() * 1.08))
    ax_lam.set_xticks(k, [n.replace("-methine", "") for n in names])
    ax_lam.set_xlabel("CH groups in the chain" if names[0].endswith("methine") else "C=C bonds", labelpad=1)
    ax_lam.set_ylabel(r"$\lambda_{max}$ (nm)")
    ax_lam.set_title("longer box → smaller gap → redder light", fontsize=10)
    ax_lam.legend(fontsize=7, loc="upper left")

    # --- colour swatches: what each dye looks like (white light minus its absorption band) ---
    ax_sw.set_xlim(-0.6, B - 0.4)
    ax_sw.set_ylim(-0.75, 1.0)
    ax_sw.axis("off")
    ax_sw.set_title("colour of the solution", fontsize=10)
    frames_sw = []
    for b in range(B):
        ax_sw.add_patch(Rectangle((b - 0.38, 0.0), 0.76, 0.8, color=colour[b], ec="#9ca3af"))
        ax_sw.text(b, -0.25, f"{lam_num[b]:.0f} nm", ha="center", fontsize=7.5, color=_rgb(lam_num[b]))
        frame = Rectangle((b - 0.45, -0.07), 0.9, 0.94, fill=False, ec="#f59e0b", lw=2.5, visible=False)
        ax_sw.add_patch(frame)
        frames_sw.append(frame)
    ax_sw.text((B - 1) / 2, -0.62, "UV absorbers look colourless", ha="center", fontsize=7, color="#6b7280")

    def draw_box(b: int, f: float):
        ax_box.cla()
        ax_box.set_xlim(-0.08, L.max() + 0.08)
        ax_box.set_ylim(-0.12 * e_top, e_top)
        ax_box.set_xlabel("position along the chain (nm)", labelpad=1)
        ax_box.set_ylabel("energy (eV)")
        ax_box.set_title(f"{names[b]}: {n_el[b]} π electrons in L = {L[b]:.2f} nm", fontsize=10)
        ax_box.plot([0, 0, L[b], L[b]], [e_top, 0, 0, e_top], color="#374151", lw=2)  # the box walls
        h = homo[b]
        for n in range(h + 2):  # filled levels and the LUMO
            ax_box.plot([0, L[b]], [E[b, n]] * 2, color="#9ca3af" if n <= h else "#dc2626", lw=1)
            if n < h:
                ax_box.plot([0.42 * L[b], 0.58 * L[b]], [E[b, n]] * 2, "o", ms=4, color="#2563eb")
        dE = E[b, h + 1] - E[b, h]
        for n, c in ((h, "#2563eb"), (h + 1, "#dc2626")):  # HOMO and LUMO wavefunctions, drawn at their energy
            ax_box.plot(x[b], E[b, n] + 0.35 * dE * psi[b, :, n] / np.abs(psi[b, :, n]).max(), color=c, lw=1.2)
        # one HOMO electron stays, the other jumps to the LUMO as f goes 0 → 1
        ax_box.plot([0.42 * L[b]], [E[b, h]], "o", ms=4, color="#2563eb")
        ax_box.plot([0.58 * L[b]], [E[b, h] + f * dE], "o", ms=5, color="#2563eb")
        if f > 0.05:
            ax_box.annotate(
                "", xy=(0.7 * L[b], E[b, h] + f * dE), xytext=(0.7 * L[b], E[b, h]),
                arrowprops={"arrowstyle": "->", "color": _rgb(lam_num[b]), "lw": 2},
            )  # fmt: skip
        ax_box.text(
            0.74 * L[b], E[b, h] + 0.5 * dE, f"{lam_num[b]:.0f} nm", fontsize=8, va="center", color=_rgb(lam_num[b])
        )
        ax_box.text(L[b] + 0.03, E[b, h], "HOMO", fontsize=7, va="center", color="#2563eb")
        ax_box.text(L[b] + 0.03, E[b, h + 1], "LUMO", fontsize=7, va="center", color="#dc2626")
        # the conjugated chain drawn as a zigzag inside the bottom of the box
        xs = end * bond + bond * np.arange(n_bonds[b] + 1)  # atom positions: one bond length apart
        ys = np.where(np.arange(n_bonds[b] + 1) % 2 == 0, -0.04, -0.09) * e_top
        ax_box.plot(xs, ys, "-o", ms=3, lw=1, color="#6b7280")

    def update(step: int):
        b, i = divmod(step, FRAMES_PER_MOLECULE)
        f = min(1.0, i / (0.6 * FRAMES_PER_MOLECULE))
        draw_box(b, f)
        mark.set_data([b], [lam_num[b]])
        for j, frame in enumerate(frames_sw):
            frame.set_visible(j == b)

    fig.tight_layout()
    n_steps = B * FRAMES_PER_MOLECULE
    return save_gif(fig, update, frame_indices(n_steps, n_steps), path, fps=8)


def _rainbow() -> np.ndarray:
    """A vertical rainbow image (380–780 nm) for the background of the λ panel."""
    from .equation import wavelength_rgb

    return wavelength_rgb(torch.linspace(380, 780, 120))[:, None, :].numpy()


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'series': 'cyanine',
    'n_max': 5,
    'bond': 0.139,
    'end': 1.0,
    'V_alt': 0.0,
    'n_points': 400
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.physchem.particle_box_dyes")]:
    del sys.modules[name]
main = importlib.import_module("problems.physchem.particle_box_dyes.main")
problem_dir = Path("problems/physchem/particle_box_dyes")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
